# 04 — Metrics Evaluation & Benchmark Runner
**Owner: Poornam**  
**Pipeline:** Noisy image → OCR service → Compression service → Metrics

### What this notebook does
1. Mounts Drive and clones the repo
2. Imports `metrics.py` and `benchmarks/latency.py`
3. Lets you run metrics standalone (no services needed) for unit checks
4. Runs the full end-to-end benchmark once both services are live
5. Prints markdown tables ready to paste into `REPORT.md`

---
### When to run each section
| Section | When |
|---|---|
| Setup + standalone tests | Now (Block A) |
| Per-noise CER table | After Anuj hits 95% val accuracy (h11) |
| Full benchmark | After both services are running (h15) |
| Final tables for REPORT.md | After benchmark run on TE split (h16) |

## 0. Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL    = 'https://github.com/sampreethkabadi/luddy-hack.git'  # update if URL differs
REPO_DIR    = '/content/luddy-hack'
BRANCH      = 'main'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies
!pip install requests --quiet

## 1. Import metrics — standalone (no services needed)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'benchmarks'))

from benchmarks.metrics import (
    cer,
    cer_accuracy,
    compression_ratio,
    shannon_entropy,
    encoding_efficiency,
    aggregate_cer_by_noise,
)

print('metrics.py imported successfully')

## 2. Standalone metric checks
Run these now to confirm everything works before services are up.

In [ ]:
# --- CER sanity check ---
ref  = 'the cat sat on the mat'
hyp1 = 'the cat sat on the mat'   # perfect
hyp2 = 'the cat sat on the map'   # 1 substitution

print('CER (perfect):       ', round(cer(ref, hyp1), 4), '← expect 0.0')
print('CER (1 sub):         ', round(cer(ref, hyp2), 4), '← expect ~0.0455')
print('Accuracy (1 sub):    ', round(cer_accuracy(ref, hyp2), 4))

In [ ]:
# --- Entropy + efficiency sanity check ---
text = 'AABBBCCCC'
print('Entropy (AABBBCCCC): ', round(shannon_entropy(text), 4), '← expect ~1.5305')
print('Entropy (AAAA):      ', round(shannon_entropy('AAAA'), 4),  '← expect 0.0')
print('Entropy (ABCD):      ', round(shannon_entropy('ABCD'), 4),  '← expect 2.0')
print('Efficiency (15 bits):', round(encoding_efficiency(text, 15), 4), '← expect ~0.917')

In [ ]:
# --- Aggregate CER by noise type ---
# Replace with real OCR results once Anuj's model is running
sample_results = [
    {'noise_type': 'f', 'reference': 'hello world', 'hypothesis': 'hello world'},
    {'noise_type': 'f', 'reference': 'hello world', 'hypothesis': 'helo world'},
    {'noise_type': 'w', 'reference': 'test line',   'hypothesis': 'test line'},
    {'noise_type': 'c', 'reference': 'coffee stain','hypothesis': 'coffe stain'},
    {'noise_type': 'p', 'reference': 'footprint',   'hypothesis': 'footprint'},
]

table = aggregate_cer_by_noise(sample_results)
print(f"{'Noise':<10} {'Mean CER':>10} {'Accuracy':>10} {'N':>4}")
print('-' * 36)
for noise, stats in table.items():
    if stats['n'] > 0:
        print(f"{noise:<10} {stats['mean_cer']:>10.4f} {stats['mean_accuracy']:>10.4f} {stats['n']:>4}")

## 3. Compute CER from Anuj's validation results
**Run this after h11 checkpoint** — paste Anuj's val output into `val_results` below.

In [ ]:
# Paste Anuj's validation results here once available.
# Each dict needs: noise_type, reference (ground truth), hypothesis (model output)

val_results = [
    # {'noise_type': 'f', 'reference': '...', 'hypothesis': '...'},
    # {'noise_type': 'w', 'reference': '...', 'hypothesis': '...'},
    # {'noise_type': 'c', 'reference': '...', 'hypothesis': '...'},
    # {'noise_type': 'p', 'reference': '...', 'hypothesis': '...'},
]

if val_results:
    cer_table = aggregate_cer_by_noise(val_results)

    # Markdown table for REPORT.md
    print('### OCR Accuracy by Noise Type\n')
    print('| Noise type | N | Mean CER | Mean Accuracy |')
    print('|---|---|---|---|')
    noise_labels = {'f': 'Folded', 'w': 'Wrinkled', 'c': 'Coffee', 'p': 'Footprints', 'overall': 'Overall'}
    for noise, stats in cer_table.items():
        if stats['n'] > 0:
            label = noise_labels.get(noise, noise)
            print(f"| {label:<10} | {stats['n']} | {stats['mean_cer']:.4f} | {stats['mean_accuracy']:.4f} |")
else:
    print('No val_results yet — fill in after h11 checkpoint.')

## 4. Full end-to-end benchmark
**Run this after h15 checkpoint** — both services must be running.

To start services locally in Colab, open two separate terminal cells:
```bash
# Terminal 1 — OCR service
cd /content/luddy-hack && uvicorn stage1_ocr.service:app --port 8001

# Terminal 2 — Compression service  
cd /content/luddy-hack && uvicorn stage2_huffman.service:app --port 8002
```

In [ ]:
# Health check — confirm both services are up before running benchmark
import requests

OCR_URL  = 'http://localhost:8001'
COMP_URL = 'http://localhost:8002'

try:
    r1 = requests.get(f'{OCR_URL}/health',  timeout=3)
    print('OCR service: ', r1.json())
except Exception as e:
    print('OCR service OFFLINE:', e)

try:
    r2 = requests.get(f'{COMP_URL}/health', timeout=3)
    print('Comp service:', r2.json())
except Exception as e:
    print('Comp service OFFLINE:', e)

In [ ]:
# Run the full benchmark
# Only run once both health checks above show {"status": "ok"}

from benchmarks.latency import run_in_colab

results = run_in_colab(
    image_dir   = f'{REPO_DIR}/data/SimulatedNoisyOffice/test',
    labels_path = f'{REPO_DIR}/data/labels.json',
    ocr_url     = OCR_URL,
    comp_url    = COMP_URL,
    n           = 50,
    algo        = 'fgk',
    out_path    = f'{REPO_DIR}/benchmarks/results/run_te_fgk.json',
    split       = 'TE',
)

## 5. Print markdown tables for REPORT.md
Run after the benchmark cell above completes.

In [ ]:
# Tables are already printed by run_in_colab above.
# This cell re-prints them cleanly from the saved JSON — useful if
# you want to re-run just the reporting step without hitting services again.

import json
from benchmarks.latency import print_summary_table

RESULTS_PATH = f'{REPO_DIR}/benchmarks/results/run_te_fgk.json'

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        saved = json.load(f)
    print_summary_table(saved['summary'])
    print(f"\nTotal samples: {saved['config']['n_completed']}")
    print(f"Failed:        {saved['config']['n_failed']}")
else:
    print('No results file yet — run Section 4 first.')

## 6. Save results to Drive (backup)
Run after benchmark to back up results to your Drive.

In [ ]:
import shutil

DRIVE_BACKUP = '/content/drive/MyDrive/luddy_hack_results/'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

src = f'{REPO_DIR}/benchmarks/results/'
if os.path.exists(src):
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), DRIVE_BACKUP)
        print(f'Backed up: {f}')
else:
    print('No results to back up yet.')